# QUERY ENGINE - Used for retrieving information and answering individual questions from your data.

**What Each One Does**

| Query Engine                  | Main job                                           | 
| ----------------------------- | -------------------------------------------------- | 
| **RetrieverQueryEngine**      | Retrieve relevant chunks → answer                  |                     
| **CitationQueryEngine**       | Retrieve → answer with source/citation information |                     
| **RouterQueryEngine**         | Select the appropriate query engine                |                     
| **MultiStepQueryEngine**      | Solve a query through multiple steps               |                     
| **KnowledgeGraphQueryEngine** | Query entities and relationships in a graph        |      
| **Summary approach**          | Summarize document/index content                   |                    
| **SubQuestionQueryEngine**    | Break complex question into sub-questions          |                     
| **CustomQueryEngine**         | Build your own query-processing logic              |                      


# IMPORTS

In [3]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from dotenv import load_dotenv

In [4]:
import logging
import warnings

warnings.filterwarnings("ignore", message="Mixing V1 models and V2 models")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.disable(logging.WARNING)

# CONFIG

In [5]:
load_dotenv()
llm = OpenAI(model="gpt-4o-mini")

# LOAD PDF

In [6]:
documents = SimpleDirectoryReader(input_files=["data/HopeAI.pdf"]).load_data()
print("PDF Loaded")

PDF Loaded


# CREATE EMBEDDING + INDEX + RETRIEVER

In [7]:
embed_model = OpenAIEmbedding(model="text-embedding-3-small")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
retriever = index.as_retriever(similarity_top_k=3)

<h1 style="color: #EA580C;">1. RETRIEVER QUERY ENGINE</h1>

In [9]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=llm
)
response = query_engine.query("What technologies are taught at HopeAI?")

print("\nANSWER")
print(response)


ANSWER
The technologies taught include Python, various libraries such as Numpy, Pandas, Matplotlib, Sklearn, and Seaborn, as well as concepts in Data Science, Machine Learning (including Regression, Classification, and Clustering), Deep Learning (including Artificial Neural Networks, Convolutional Neural Networks, RNNs, and LSTMs), Natural Language Processing (NLP), and web development with Django. Additionally, cloud technologies such as AWS, GCP, and Azure are covered, along with advanced AI topics like Transformer Architecture, LLM, and AI Agents.


<h1 style="color: #EA580C;">2. CITATION QUERY ENGINE</h1>

**This is similar to the Retriever Query Engine, but the response is designed to retain citation/source information**

In [11]:
from llama_index.core.query_engine import CitationQueryEngine

query_engine = CitationQueryEngine.from_args(
    index=index,
    llm=llm,
    citation_chunk_size=512
)
response = query_engine.query("What technologies are taught at HopeAI?")

print("/nANSWER WITH CITATIONS")
print(response)

/nANSWER WITH CITATIONS
Hope AI teaches a variety of technologies related to Artificial Intelligence and Data Science, including Python basics, libraries such as Numpy, Pandas, Matplotlib, and Sklearn, as well as machine learning techniques like regression, classification, and clustering. Additionally, the curriculum covers advanced topics such as artificial neural networks, deep learning, natural language processing, and web development with Django [2].


<h1 style="color: #EA580C;">3. ROUTER QUERY ENGINE</h1>

**A Router Query Engine chooses between multiple query engines depending on the question.**

<div align="center">
    <img src="images/router.png" width="700" height="2000">
</div>

In [12]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.core.selectors import LLMSingleSelector

# CREATE TWO QUERY ENGINES

course_engine = index.as_query_engine(similarity_top_k=3)
general_engine = index.as_query_engine(similarity_top_k=3)

# WRAP THEM AS TOOLS

course_tool = QueryEngineTool.from_defaults(
    query_engine=course_engine,
    description=("Use this engine for questions about HopeAI courses, training programs, curriculum and technologies.")
)

general_tool = QueryEngineTool.from_defaults(
    query_engine=general_engine,
    description=("Use this engine for general questions about HopeAI institute, facilities, support and other information."))

# CREATE ROUTER

query_engine = RouterQueryEngine.from_defaults(
    query_engine_tools=[course_tool, general_tool],
    llm=llm,
    selector=LLMSingleSelector.from_defaults(llm=llm),
    verbose=True
)
response = query_engine.query("What courses does HopeAI offer, and what support or facilities are provided to students?")

print("ANSWER")
print(response)

2026-09-25 13:28:12,217 - INFO - Selecting query engine 0: The question specifically asks about the courses offered by HopeAI, which is covered by choice 1..


Selecting query engine 0: The question specifically asks about the courses offered by HopeAI, which is covered by choice 1..


2026-09-25 13:28:16,252 - INFO - Selecting query engine 1: The question also inquires about support and facilities provided to students, which is addressed by choice 2..


Selecting query engine 1: The question also inquires about support and facilities provided to students, which is addressed by choice 2..


2026-09-25 13:28:19,463 - INFO - Combining responses from multiple query engines.


ANSWER
HopeAI offers courses in Artificial Intelligence and Data Science. Students receive comprehensive knowledge and practical skills in these fields, along with mentorship from industry experts. The learning format is self-paced, allowing for customized career paths based on individual work experience and academic history. Support includes 24/7 doubt clarification via WhatsApp, assistance with resume preparation and LinkedIn growth, and personalized mentorship. The courses are accessible from anywhere, providing flexibility to fit busy schedules and the convenience of learning from home without travel. Additional features include intensive placement assistance, a course duration of 2 hours per day for 6 months, extended support for 3 years, lifetime access to course materials, and real-time scenario-based capstone projects tailored to individual interests.


<h1 style="color: #EA580C;">4. MULTI-STEP QUERY ENGINE</h1>

**A Multi-Step Query Engine handles a question through multiple query steps instead of simply performing one retrieval.**

<div align="center">
    <img src="images/multistep.png" width="700" height="2000">
</div>

In [14]:
from llama_index.core.query_engine import MultiStepQueryEngine
from llama_index.core.indices.query.query_transform import StepDecomposeQueryTransform

# CREATE BASE QUERY ENGINE
base_query_engine = index.as_query_engine(similarity_top_k=3)

# CREATE STEP DECOMPOSER
step_decompose = StepDecomposeQueryTransform(llm)

# CREATE MULTI-STEP ENGINE
query_engine = MultiStepQueryEngine(
    query_engine=base_query_engine,
    query_transform=step_decompose,
    index_summary=("This index contains information about HopeAI courses and services.")
)
response = query_engine.query("What courses does HopeAI offer and what technologies are covered in those courses?")

print("FINAL ANSWER")
print(response)

FINAL ANSWER
HopeAI offers courses that cover a range of technologies including Python, Machine Learning, Natural Language Processing (NLP), Time Series Analysis, Deep Learning, Cloud services like AWS, GCP, Azure, and advanced AI topics such as Transformer Architecture, LLM, AI Agents, and RAG.


<h1 style="color: #EA580C;">5. KNOWLEDGE GRAPH QUERY ENGINE</h1>

**How it works?**<br>
**User Query: What technologies are taught in HopeAI courses, and how are they related to the training programs?**

<div align="center">
    <img src="images/kg.png" width="700" height="2000">
</div>

In [18]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core.storage.storage_context import StorageContext
from llama_index.core.graph_stores import SimpleGraphStore

# CREATE GRAPH STORE
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

# CREATE KNOWLEDGE GRAPH
index = KnowledgeGraphIndex.from_documents(
    documents,
    storage_context=storage_context,
    max_triplets_per_chunk=3,
    embed_model=embed_model,
    llm=llm
)

# CREATE QUERY ENGINE
query_engine = index.as_query_engine(include_text=True, response_mode="tree_summarize")
response = query_engine.query("What technologies are taught in HopeAI courses, and how are they related to the training programs?")

print("ANSWER")
print(response)

ANSWER
HopeAI courses teach a variety of technologies such as artificial intelligence, machine learning, data science, and programming languages like Python. These technologies are integrated into the training programs to provide students with a comprehensive understanding of AI concepts and applications.


<h1 style="color: #EA580C;">6. SUMMARY QUERY ENGINE</h1>

<div align="center">
    <img src="images/summary.png" width="400" height="500">
</div>

In [20]:
from llama_index.core import SummaryIndex

# CREATE SUMMARY INDEX
index = SummaryIndex.from_documents(documents)

# CREATE QUERY ENGINE
query_engine = index.as_query_engine(response_mode="tree_summarize", llm=llm)

response = query_engine.query("Give me a summary of the HopeAI course.")

print("ANSWER")
print(response)

ANSWER
Hope AI offers an E-Learning platform focused on Artificial Intelligence and Data Science, providing comprehensive courses designed to empower individuals with the necessary knowledge and skills for success in these fields. Founded in 2019 and converted to a private limited company in December 2022, the platform caters to a diverse audience, including IT professionals, non-IT individuals, career changers, and fresh graduates.

The courses are self-paced and include personalized mentorship, ensuring tailored guidance for each student. Key features include 24/7 support for doubt clarification, assistance with resume preparation, and LinkedIn growth strategies. The curriculum covers a wide range of topics, including Python programming, data science fundamentals, machine learning techniques, deep learning, natural language processing, and web development with AI integration. 

Students engage in real-time scenario-based capstone projects and receive intensive placement assistance, w

<h1 style="color: #EA580C;">7. SUB QUESTION QUERY ENGINE</h1>

**A Sub Question Query Engine takes one complex question, breaks it into smaller questions, answers those separately, and combines the answers.**

**How it works?**<br>
**User Query: What courses does HopeAI offer and what technologies are taught in those courses?**

<div align="center">
    <img src="images/subquestion.png" width="700" height="1500">
</div>

In [8]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool

# CREATE INDEX
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# CREATE QUERY ENGINE
query_engine = index.as_query_engine(similarity_top_k=3)

# CREATE QUERY ENGINE TOOL

tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    description=("Information about HopeAI courses, technologies, training and services.")
)

# CREATE SUB-QUESTION ENGINE

sub_question_engine = (
    SubQuestionQueryEngine.from_defaults(
        query_engine_tools=[tool],
        llm=llm,
        use_async=False
    )
)

response = sub_question_engine.query("What courses does HopeAI offer and what technologies are taught in those courses?")

print("FINAL ANSWER")
print(response)

Generated 2 sub questions.
[query_engine_tool] Q: What courses are currently offered by HopeAI?
[query_engine_tool] A: Python Basics, Libraries in Python, Data Science, Univariate Analysis, Bivariate Analysis, Machine Learning, Advanced ML, Artificial Neural Network, Convolutional Neural Network, RNN, LSTMs, Deep Learning & Computer Vision, Time Series Analysis & Forecasting, Natural Language Processing, Web Development with Django and AI Model Integration, Cloud and DevOps (AWS, GCP, Azure), Advanced AI Topics (Transformer Architecture, LLM, AI Agents, RAG).
[query_engine_tool] Q: What technologies are included in the HopeAI courses?
[query_engine_tool] A: The technologies included in the HopeAI courses are Python, Machine Learning, Natural Language Processing (NLP), Time Series Analysis, Deep Learning, and Cloud services such as AWS, GCP, and Azure.
FINAL ANSWER
HopeAI offers a variety of courses including Python Basics, Libraries in Python, Data Science, Univariate Analysis, Bivaria

<h1 style="color: #EA580C;">8. CUSTOM QUERY ENGINE</h1>

**CustomQueryEngine allows developers to create their own query-processing workflow by defining how a user's query should be handled and how the final response should be generated.**

<div align="center">
    <img src="images/customquery.png" width="700" height="1500">
</div>

In [15]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.core import VectorStoreIndex
from llama_index.core.response_synthesizers import get_response_synthesizer

# CREATE INDEX
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# CREATE QUERY ENGINE
query_engine = index.as_query_engine(similarity_top_k=3)

# CREATE CUSTOM QUERY ENGINE
class MyQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response = self.response_synthesizer.synthesize(
            query=query_str,
            nodes=nodes
        )
        return response

# CREATE RETRIEVER
retriever = index.as_retriever(similarity_top_k=3)

# CREATE RESPONSE SYNTHESIZER
response_synthesizer = get_response_synthesizer(response_mode="compact")

# CREATE CUSTOM ENGINE
custom_engine = MyQueryEngine(retriever=retriever, response_synthesizer=response_synthesizer)

# QUERY
response = custom_engine.query("What courses are offered by Hope AI?")
print("Answer:", response)

Answer: Python Basics, Libraries in Python, Data Science, Univariate Analysis, Bivariate Analysis, Machine Learning, Advanced ML, Artificial Neural Network, Convolutional Neural Network, RNN, LSTMs, Deep Learning & Computer Vision, Time Series Analysis & Forecasting, Natural Language Processing, Web Development with Django and AI Model Integration, Cloud and DevOps (AWS, GCP, Azure), Advanced AI Topics (Transformer Architecture, LLM, AI Agents, RAG).
